# Ottawa zoning bylaw: EDA

Starter notebook for M0 exploration of the Ottawa zoning bylaw PDF.

The PDF is not in git (too big). Download it into `data/raw/` first, see the README.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import pdfplumber

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
BYLAW_PDF = RAW_DIR / "ottawa_zoning_bylaw_2026-50.pdf"  # rename to match your download

assert BYLAW_PDF.exists(), f"Download the bylaw PDF to {BYLAW_PDF.resolve()}"

## First look

In [ ]:
pdf = pdfplumber.open(BYLAW_PDF)
print(f"{len(pdf.pages)} pages")
pdf.metadata

In [ ]:
page = pdf.pages[0]
print(page.extract_text()[:2000])

## Text per page

In [ ]:
pages = pd.DataFrame(
    {"page": i + 1, "text": p.extract_text() or ""} for i, p in enumerate(pdf.pages)
)
pages["n_chars"] = pages["text"].str.len()
pages["n_chars"].describe()

## Find Part headings

In [ ]:
part_pattern = r"^(Part \d+\b.*)$"
parts = (
    pages.assign(heading=pages["text"].str.findall(part_pattern, flags=re.MULTILINE))
    .explode("heading")
    .dropna(subset=["heading"])[["page", "heading"]]
)
parts.head(30)

## Tables on a page

In [ ]:
PAGE_WITH_TABLE = 1  # change me
pdf.pages[PAGE_WITH_TABLE - 1].extract_tables()